# Business Metrics — R Reference

**Companion to:** `ds1_problem_framing.ipynb` — metric definitions, roles, and business context

**Purpose:** tidyverse recipes to compute every metric from the ds1 notes. Sample
data is generated in §0 so every cell runs standalone. One section per metric,
each ending with a `ggplot2` visualization.

**A structural note on this port:** the Python source is one long markdown cell
containing ~25 embedded ` ```python ` code fences with inline matplotlib charts.
This R version is restructured into **proper alternating markdown/code cells**
so each block actually executes under the R kernel, and every chart is
rebuilt in `ggplot2` rather than matplotlib. The metric definitions, section
structure, and §0 sample-data design are preserved exactly.

---

## §0 — Imports & Sample Data

Run this section first. All data frames defined here are reused throughout the notebook.

In [ ]:
library(dplyr)
library(tidyr)
library(lubridate)
library(ggplot2)
library(scales)
library(stringr)

set.seed(42)

# ── date range ────────────────────────────────────────────────────────────────
START <- ymd('2024-01-01')
END   <- ymd('2024-03-31')
DATES <- seq(START, END, by = "day")

# ── users ─────────────────────────────────────────────────────────────────────
N_USERS  <- 2000
channels <- c('paid_search', 'organic', 'referral', 'social')
users <- tibble::tibble(
  user_id    = 1:N_USERS,
  created_at = sample(seq(ymd('2023-10-01'), ymd('2024-03-01'), by = "day"), N_USERS, replace = TRUE),
  channel    = sample(channels, N_USERS, replace = TRUE, prob = c(0.35, 0.30, 0.20, 0.15))
)

# ── events ────────────────────────────────────────────────────────────────────
N_EVENTS   <- 30000
event_types <- c('session', 'page_view', 'add_to_cart', 'checkout', 'purchase',
                  'invite_sent', 'invite_accepted')
events <- tibble::tibble(
  user_id    = sample(users$user_id, N_EVENTS, replace = TRUE),
  event_date = sample(DATES, N_EVENTS, replace = TRUE),
  event_type = sample(event_types, N_EVENTS, replace = TRUE,
                       prob = c(0.30, 0.25, 0.15, 0.12, 0.10, 0.05, 0.03))
)

# ── orders ────────────────────────────────────────────────────────────────────
purchase_events <- events |> filter(event_type == 'purchase')
orders <- purchase_events |> select(user_id, order_date = event_date) |>
  mutate(order_id = row_number(), .before = 1) |>
  mutate(order_value = round(rlnorm(n(), meanlog = 3.5, sdlog = 0.8), 2))

# ── marketing_spend ───────────────────────────────────────────────────────────
base_spend <- c(paid_search = 500, organic = 50, referral = 150, social = 200)
marketing_spend <- expand.grid(spend_date = seq(ymd('2024-01-01'), ymd('2024-03-31'), by = "day"),
                                channel = channels, stringsAsFactors = FALSE) |>
  tibble::as_tibble() |>
  mutate(spend = round(base_spend[channel] * runif(n(), 0.8, 1.2), 2))

# ── subscriptions (SaaS) ──────────────────────────────────────────────────────
sub_users <- users |> slice_sample(n = 600) |> pull(user_id)
months_seq <- seq(ymd('2024-01-01'), ymd('2024-03-01'), by = "month")
subscriptions <- purrr::map_dfr(sub_users, function(uid) {
  fee <- sample(c(9.99, 19.99, 49.99), 1, prob = c(0.5, 0.35, 0.15))
  tibble::tibble(
    user_id = uid, month = months_seq, monthly_fee = fee,
    status = if_else(runif(length(months_seq)) < 0.05, 'churned', 'active')
  )
})

# ── ad_events ─────────────────────────────────────────────────────────────────
N_ADS <- 20000
ad_ids <- sprintf('ad_%03d', 1:10)
ad_events <- tibble::tibble(
  user_id    = sample(users$user_id, N_ADS, replace = TRUE),
  event_date = sample(DATES, N_ADS, replace = TRUE),
  ad_id      = sample(ad_ids, N_ADS, replace = TRUE),
  event_type = sample(c('impression', 'click'), N_ADS, replace = TRUE, prob = c(0.92, 0.08))
)

# ── transactions (marketplace) ────────────────────────────────────────────────
N_TXN <- 5000
transactions <- tibble::tibble(
  transaction_id    = 1:N_TXN,
  buyer_id          = sample(users$user_id, N_TXN, replace = TRUE),
  seller_id         = sample(users$user_id, N_TXN, replace = TRUE),
  transaction_date  = sample(DATES, N_TXN, replace = TRUE),
  transaction_value = round(rlnorm(N_TXN, meanlog = 4.0, sdlog = 0.9), 2)
)

# ── nps_responses ─────────────────────────────────────────────────────────────
set.seed(7)
nps_users <- users |> slice_sample(n = 800) |> pull(user_id)
nps_responses <- tibble::tibble(
  user_id       = nps_users,
  response_date = sample(DATES, length(nps_users), replace = TRUE),
  score         = sample(0:10, length(nps_users), replace = TRUE,
                          prob = c(0.03,0.02,0.03,0.04,0.05,0.06,0.08,0.09,0.15,0.20,0.25))
)

cat("Sample data ready:\n")
cat(sprintf("  users:           %s rows\n", format(nrow(users), big.mark=",")))
cat(sprintf("  events:          %s rows\n", format(nrow(events), big.mark=",")))
cat(sprintf("  orders:          %s rows\n", format(nrow(orders), big.mark=",")))
cat(sprintf("  marketing_spend: %s rows\n", format(nrow(marketing_spend), big.mark=",")))
cat(sprintf("  subscriptions:   %s rows\n", format(nrow(subscriptions), big.mark=",")))
cat(sprintf("  ad_events:       %s rows\n", format(nrow(ad_events), big.mark=",")))
cat(sprintf("  transactions:    %s rows\n", format(nrow(transactions), big.mark=",")))
cat(sprintf("  nps_responses:   %s rows\n", format(nrow(nps_responses), big.mark=",")))

---

## §1 — DAU / WAU / MAU

**Formula:**
- DAU = unique users active on a given day
- WAU = unique users active in a given ISO week
- MAU = unique users active in a given calendar month

In [ ]:
# DAU
dau <- events |> group_by(event_date) |> summarise(dau = n_distinct(user_id), .groups = "drop")

# WAU
events <- events |> mutate(year_week = floor_date(event_date, "week", week_start = 1))
wau <- events |> group_by(year_week) |> summarise(wau = n_distinct(user_id), .groups = "drop") |>
  rename(week_start = year_week)

# MAU
events <- events |> mutate(year_month = floor_date(event_date, "month"))
mau <- events |> group_by(year_month) |> summarise(mau = n_distinct(user_id), .groups = "drop")

cat("DAU (first 5 days):\n")
print(head(dau, 5))
cat("\nMAU by month:\n")
print(mau)

In [ ]:
library(patchwork)

p1 <- ggplot(dau, aes(x = event_date, y = dau)) +
  geom_line(color = '#3498DB', linewidth = 0.8, alpha = 0.9) +
  labs(title = 'Daily Active Users (DAU)', x = 'Date', y = 'Unique users') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"), panel.grid.minor = element_blank())

p2 <- ggplot(mau, aes(x = format(year_month, "%Y-%m"), y = mau)) +
  geom_col(fill = '#2ECC71', color = 'white', width = 0.5) +
  labs(title = 'Monthly Active Users (MAU)', x = 'Month', y = 'Unique users') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"), panel.grid.minor = element_blank())

p1 + p2

---

## §2 — DAU/MAU Ratio

**Formula:** DAU/MAU ratio = DAU / MAU for the same calendar month

**Interpretation:** 0.20 -> users active ~6 out of 30 days per month. Industry
benchmarks: >0.20 is reasonable, >0.50 is exceptional (WhatsApp-tier).

In [ ]:
# Add month column to dau
dau <- dau |> mutate(year_month = floor_date(event_date, "month"))

# Join DAU with MAU on year_month
dau_mau <- dau |> left_join(mau, by = "year_month") |> mutate(dau_mau_ratio = dau / mau)

print(head(dau_mau |> select(event_date, dau, mau, dau_mau_ratio), 10))
cat("\nMean DAU/MAU ratio by month:\n")
print(dau_mau |> group_by(year_month) |> summarise(mean_ratio = round(mean(dau_mau_ratio), 4), .groups = "drop"))

In [ ]:
library(slider)

# 7-day rolling average of DAU/MAU
dau_mau_sorted <- dau_mau |> arrange(event_date) |>
  mutate(rolling_ratio = slide_dbl(dau_mau_ratio, mean, .before = 6, .complete = TRUE))

ggplot(dau_mau_sorted, aes(x = event_date)) +
  geom_ribbon(aes(ymin = 0, ymax = dau_mau_ratio), alpha = 0.2, fill = '#9B59B6') +
  geom_line(aes(y = rolling_ratio), color = '#9B59B6', linewidth = 1, na.rm = TRUE) +
  geom_hline(yintercept = 0.20, color = '#E74C3C', linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = max(dau_mau_sorted$event_date), y = 0.20, label = "0.20 benchmark",
           hjust = 1, vjust = -0.5, color = '#E74C3C', size = 3) +
  labs(title = 'DAU/MAU Ratio over Time', x = 'Date', y = 'DAU/MAU ratio') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

---

## §3 — D1 / D7 / D30 Retention

**Formula:** Dn retention = users active on day n after signup / cohort size (users who signed up on day 0)

In [ ]:
# Cohort: signup date per user
cohort <- users |> select(user_id, created_at) |> mutate(cohort_month = floor_date(created_at, "month"))

# All event dates
activity <- events |> select(user_id, event_date) |> distinct()

# Join and compute days since signup
merged <- cohort |> left_join(activity, by = "user_id") |>
  mutate(day_number = as.numeric(event_date - created_at, units = "days"))

# Cohort sizes
cohort_sizes <- cohort |> group_by(cohort_month) |> summarise(cohort_size = n_distinct(user_id), .groups = "drop")

# Retention for each day 0-30
retention_rows <- purrr::map_dfr(0:30, function(day) {
  active_on_day <- merged |> filter(day_number == day) |>
    group_by(cohort_month) |> summarise(retained = n_distinct(user_id), .groups = "drop")
  cohort_sizes |> left_join(active_on_day, by = "cohort_month") |>
    mutate(retained = replace_na(retained, 0), day_number = day,
           retention_rate = retained / cohort_size)
})

retention <- retention_rows

# Summary: D1, D7, D30 per cohort month
pivot <- retention |> filter(day_number %in% c(1, 7, 30)) |>
  select(cohort_month, day_number, retention_rate) |>
  pivot_wider(names_from = day_number, values_from = retention_rate, names_prefix = "D") |>
  arrange(cohort_month)
print(pivot |> mutate(across(starts_with("D"), ~round(.x, 4))))

In [ ]:
colors <- c('#3498DB', '#2ECC71', '#E74C3C', '#F39C12')

ggplot(retention, aes(x = day_number, y = retention_rate, color = factor(cohort_month))) +
  geom_line(linewidth = 1, alpha = 0.85) +
  geom_vline(xintercept = c(1, 7, 30), color = "grey50", linetype = "dotted", linewidth = 0.5) +
  annotate("text", x = c(1, 7, 30), y = 0.02, label = c("D1", "D7", "D30"), size = 3, color = "grey50") +
  scale_color_manual(values = colors, labels = function(x) format(as.Date(x), "%Y-%m"), name = "Cohort month") +
  scale_y_continuous(labels = percent_format()) +
  labs(title = 'Retention Curves by Cohort Month (D0-D30)', x = 'Days since signup', y = 'Retention rate') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

---

## §4 — K-factor

**Formula:** K-factor = (invites sent per user) x (invite conversion rate)
- Invites sent per user = total invite_sent events / unique inviters
- Invite conversion rate = invite_accepted / invite_sent

In [ ]:
inv <- events |> filter(event_type %in% c('invite_sent', 'invite_accepted'))

total_sent     <- sum(inv$event_type == 'invite_sent')
total_accepted <- sum(inv$event_type == 'invite_accepted')
total_inviters <- inv |> filter(event_type == 'invite_sent') |> pull(user_id) |> n_distinct()

invites_per_user <- if (total_inviters > 0) total_sent / total_inviters else 0
invite_conv_rate <- if (total_sent > 0) total_accepted / total_sent else 0
k_factor         <- invites_per_user * invite_conv_rate

cat(sprintf("Total invites sent:       %s\n", format(total_sent, big.mark=",")))
cat(sprintf("Total invites accepted:   %s\n", format(total_accepted, big.mark=",")))
cat(sprintf("Unique inviters:          %s\n", format(total_inviters, big.mark=",")))
cat(sprintf("Invites per user:         %.3f\n", invites_per_user))
cat(sprintf("Invite conversion rate:   %.3f\n", invite_conv_rate))
cat(sprintf("K-factor:                 %.3f\n", k_factor))
cat(sprintf("Viral?                    %s\n", if (k_factor > 1) "Yes (>1)" else "No (<1)"))

In [ ]:
# Monthly K-factor trend
monthly_inv <- events |> filter(event_type %in% c('invite_sent', 'invite_accepted')) |>
  mutate(year_month = floor_date(event_date, "month"))

k_monthly <- monthly_inv |> group_by(year_month) |> summarise(
  sent     = sum(event_type == 'invite_sent'),
  accepted = sum(event_type == 'invite_accepted'),
  inviters = n_distinct(user_id[event_type == 'invite_sent']),
  .groups = "drop"
) |> mutate(
  invites_per_user  = if_else(inviters > 0, sent / inviters, 0),
  invite_conv_rate  = if_else(sent > 0, accepted / sent, 0),
  k_factor          = invites_per_user * invite_conv_rate
)

ggplot(k_monthly, aes(x = format(year_month, "%Y-%m"), y = k_factor)) +
  geom_col(fill = '#E67E22', color = 'white', width = 0.4) +
  geom_hline(yintercept = 1.0, color = '#E74C3C', linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = 1, y = 1.0, label = "K=1 (viral threshold)", hjust = 0, vjust = -0.5, color = '#E74C3C', size = 3) +
  labs(title = 'K-factor by Month', x = 'Month', y = 'K-factor') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

---

## §5 — NPS (Net Promoter Score)

**Formula:** NPS = % Promoters (9-10) - % Detractors (0-6)
- Passives (7-8) are in the denominator but excluded from both sides.

In [ ]:
nps_responses <- nps_responses |> mutate(nps_group = case_when(
  score >= 9 ~ 'promoter',
  score <= 6 ~ 'detractor',
  .default = 'passive'
))

counts <- table(nps_responses$nps_group)
total  <- nrow(nps_responses)

pct_promoters  <- as.numeric(counts['promoter'])  / total * 100
pct_detractors <- as.numeric(counts['detractor']) / total * 100
pct_passives   <- as.numeric(counts['passive'])   / total * 100
nps_score      <- pct_promoters - pct_detractors

cat(sprintf("Total responses:  %s\n", format(total, big.mark=",")))
cat(sprintf("Promoters:        %s  (%.1f%%)\n", format(counts['promoter'], big.mark=","), pct_promoters))
cat(sprintf("Passives:         %s   (%.1f%%)\n", format(counts['passive'], big.mark=","), pct_passives))
cat(sprintf("Detractors:       %s (%.1f%%)\n", format(counts['detractor'], big.mark=","), pct_detractors))
cat(sprintf("\nNPS Score:        %.1f\n", nps_score))

In [ ]:
library(patchwork)

p1 <- ggplot(nps_responses, aes(x = score)) +
  geom_histogram(binwidth = 1, fill = '#3498DB', color = 'white', alpha = 0.85, boundary = -0.5) +
  annotate("rect", xmin = -0.5, xmax = 6.5, ymin = 0, ymax = Inf, alpha = 0.08, fill = '#E74C3C') +
  annotate("rect", xmin = 6.5, xmax = 8.5, ymin = 0, ymax = Inf, alpha = 0.08, fill = '#F39C12') +
  annotate("rect", xmin = 8.5, xmax = 10.5, ymin = 0, ymax = Inf, alpha = 0.08, fill = '#2ECC71') +
  labs(title = 'NPS Score Distribution', x = 'Score', y = 'Count') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

breakdown <- tibble::tibble(
  label = factor(c('Promoters', 'Passives', 'Detractors'), levels = c('Promoters', 'Passives', 'Detractors')),
  value = c(pct_promoters, pct_passives, pct_detractors)
)
p2 <- ggplot(breakdown, aes(x = label, y = value, fill = label)) +
  geom_col(color = 'white', width = 0.5) +
  geom_text(aes(label = sprintf("%.1f%%", value)), vjust = -0.5, fontface = "bold") +
  scale_fill_manual(values = c('#2ECC71', '#F39C12', '#E74C3C'), guide = "none") +
  labs(title = sprintf('NPS Breakdown  (NPS = %.1f)', nps_score), x = NULL, y = '% of respondents') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

p1 + p2

---

## §6 — Conversion Rate

**Formula:** Conversion rate = unique converters / unique visitors

In [ ]:
funnel_steps <- c('session', 'page_view', 'add_to_cart', 'checkout', 'purchase')

funnel_df <- tibble::tibble(
  step  = funnel_steps,
  users = sapply(funnel_steps, function(s) events |> filter(event_type == s) |> pull(user_id) |> n_distinct())
) |> mutate(
  step_conv_rate    = users / lag(users),
  overall_conv_rate = users / users[1]
)

print(funnel_df |> mutate(across(c(step_conv_rate, overall_conv_rate), ~round(.x, 4))))
cat(sprintf("\nTop-of-funnel -> Purchase conversion: %.1f%%\n",
            funnel_df$overall_conv_rate[nrow(funnel_df)] * 100))

In [ ]:
library(patchwork)

funnel_df_plot <- funnel_df |> mutate(step = factor(step, levels = rev(funnel_steps)))
colors_f <- c('#2980B9', '#3498DB', '#5DADE2', '#85C1E9', '#AED6F1')

p1 <- ggplot(funnel_df_plot, aes(x = step, y = users, fill = step)) +
  geom_col(color = 'white', width = 0.7) +
  geom_text(aes(label = format(users, big.mark = ",")), hjust = -0.1, size = 3) +
  scale_fill_manual(values = rev(colors_f), guide = "none") +
  coord_flip() +
  labs(title = 'Conversion Funnel -- Unique Users', x = NULL, y = 'Unique users') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

monthly_conv <- events |> mutate(year_month = floor_date(event_date, "month")) |>
  group_by(year_month) |> summarise(
    visitors   = n_distinct(user_id[event_type == 'session']),
    converters = n_distinct(user_id[event_type == 'purchase']),
    .groups = "drop"
  ) |> mutate(conversion_rate = converters / na_if(visitors, 0))

p2 <- ggplot(monthly_conv, aes(x = format(year_month, "%Y-%m"), y = conversion_rate * 100, group = 1)) +
  geom_line(color = '#2ECC71', linewidth = 1.2) +
  geom_point(color = '#2ECC71', size = 3) +
  labs(title = 'Monthly Conversion Rate', x = 'Month', y = 'Conversion rate (%)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

p1 + p2

---

## §7 — LTV (Lifetime Value)

**Formula:** LTV = avg order value x purchase frequency x avg customer lifespan (years)

In [ ]:
# Per-customer aggregates
cust_orders <- orders |> group_by(user_id) |> summarise(
  order_count   = n(),
  total_revenue = sum(order_value),
  avg_order_val = mean(order_value),
  first_order   = min(order_date),
  last_order    = max(order_date),
  .groups = "drop"
) |> mutate(
  lifespan_days  = as.numeric(last_order - first_order, units = "days"),
  lifespan_years = lifespan_days / 365.0
)

# Overall LTV
avg_order_val <- mean(cust_orders$avg_order_val)
avg_frequency <- mean(cust_orders$order_count)
avg_lifespan  <- mean(cust_orders$lifespan_years)
ltv           <- avg_order_val * avg_frequency * avg_lifespan

cat(sprintf("Avg order value:          $%.2f\n", avg_order_val))
cat(sprintf("Avg purchase frequency:   %.2f orders\n", avg_frequency))
cat(sprintf("Avg customer lifespan:    %.3f years (%.1f days)\n", avg_lifespan, avg_lifespan*365))
cat(sprintf("LTV:                      $%.2f\n", ltv))

# LTV by channel
cust_channel <- cust_orders |> left_join(users |> select(user_id, channel), by = "user_id")
ltv_channel <- cust_channel |> group_by(channel) |> summarise(
  customers      = n(),
  avg_order_val  = mean(avg_order_val),
  avg_frequency  = mean(order_count),
  avg_lifespan   = mean(lifespan_years),
  .groups = "drop"
) |> mutate(ltv = avg_order_val * avg_frequency * avg_lifespan)

cat("\nLTV by channel:\n")
print(ltv_channel |> select(channel, customers, avg_order_val, avg_frequency, ltv) |>
        mutate(across(where(is.numeric), ~round(.x, 2))) |> arrange(desc(ltv)))

In [ ]:
library(patchwork)

p1 <- ggplot(cust_orders, aes(x = total_revenue)) +
  geom_histogram(bins = 40, fill = '#8E44AD', color = 'white', alpha = 0.85) +
  geom_vline(xintercept = mean(cust_orders$total_revenue), color = '#E74C3C', linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = mean(cust_orders$total_revenue), y = Inf,
           label = sprintf("Mean = $%.0f", mean(cust_orders$total_revenue)), vjust = 2, hjust = -0.1, color = '#E74C3C', size = 3) +
  labs(title = 'Customer Lifetime Revenue Distribution', x = 'Total revenue per customer', y = 'Count') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

ltv_sorted <- ltv_channel |> arrange(ltv) |> mutate(channel = factor(channel, levels = channel))
p2 <- ggplot(ltv_sorted, aes(x = channel, y = ltv)) +
  geom_col(fill = '#8E44AD', color = 'white', alpha = 0.85, width = 0.6) +
  geom_text(aes(label = sprintf("$%.2f", ltv)), hjust = -0.1, size = 3) +
  coord_flip() +
  labs(title = 'LTV by Acquisition Channel', x = NULL, y = 'LTV ($)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

p1 + p2

---

## §8 — CAC (Customer Acquisition Cost)

**Formula:** CAC = total acquisition spend / new customers acquired (same period)

In [ ]:
# Monthly spend
monthly_spend <- marketing_spend |> mutate(year_month = floor_date(spend_date, "month")) |>
  group_by(year_month) |> summarise(total_spend = sum(spend), .groups = "drop")

# Monthly new customers
users <- users |> mutate(year_month = floor_date(created_at, "month"))
monthly_new <- users |> group_by(year_month) |> summarise(new_customers = n(), .groups = "drop")

cac_monthly <- monthly_spend |> inner_join(monthly_new, by = "year_month") |>
  mutate(cac = total_spend / new_customers)

print(cac_monthly |> mutate(across(where(is.numeric), ~round(.x, 2))))

# CAC by channel
channel_spend <- marketing_spend |> group_by(channel) |> summarise(total_spend = sum(spend), .groups = "drop")
channel_new <- users |> filter(created_at >= ymd('2024-01-01')) |>
  group_by(channel) |> summarise(new_customers = n(), .groups = "drop")
cac_channel <- channel_spend |> inner_join(channel_new, by = "channel") |> mutate(cac = total_spend / new_customers)
cat("\nCAC by channel:\n")
print(cac_channel |> arrange(cac) |> mutate(across(where(is.numeric), ~round(.x, 2))))

In [ ]:
library(patchwork)

p1 <- ggplot(cac_monthly, aes(x = format(year_month, "%Y-%m"), y = cac, group = 1)) +
  geom_line(color = '#E74C3C', linewidth = 1.2) +
  geom_point(color = '#E74C3C', size = 3) +
  labs(title = 'Monthly CAC', x = 'Month', y = 'CAC ($)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

cac_sorted <- cac_channel |> arrange(cac) |> mutate(channel = factor(channel, levels = channel))
p2 <- ggplot(cac_sorted, aes(x = channel, y = cac)) +
  geom_col(fill = '#E74C3C', color = 'white', alpha = 0.85, width = 0.6) +
  geom_text(aes(label = sprintf("$%.0f", cac)), hjust = -0.1, size = 3) +
  coord_flip() +
  labs(title = 'CAC by Acquisition Channel', x = NULL, y = 'CAC ($)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

p1 + p2

---

## §9 — LTV/CAC Ratio

**Formula:** LTV/CAC = LTV / CAC per acquisition channel

**Benchmark:** >= 3x healthy, 1-3x break-even, < 1x unsustainable.

In [ ]:
ltv_cac <- ltv_channel |> select(channel, ltv) |> inner_join(cac_channel |> select(channel, cac), by = "channel") |>
  mutate(
    ltv_cac_ratio = ltv / cac,
    status = case_when(
      ltv_cac_ratio >= 3 ~ 'Healthy (>=3x)',
      ltv_cac_ratio >= 1 ~ 'Break-even (1-3x)',
      .default = 'Unsustainable (<1x)'
    )
  )
print(ltv_cac |> mutate(across(where(is.numeric), ~round(.x, 2))))

In [ ]:
ltv_cac <- ltv_cac |> mutate(bar_color = case_when(
  ltv_cac_ratio >= 3 ~ '#2ECC71',
  ltv_cac_ratio >= 1 ~ '#F39C12',
  .default = '#E74C3C'
))

ggplot(ltv_cac, aes(x = channel, y = ltv_cac_ratio, fill = bar_color)) +
  geom_col(color = 'white', width = 0.5) +
  geom_text(aes(label = sprintf("%.2fx", ltv_cac_ratio)), vjust = -0.5, fontface = "bold", size = 3.5) +
  geom_hline(yintercept = 3.0, color = '#2ECC71', linetype = "dashed", linewidth = 0.8) +
  geom_hline(yintercept = 1.0, color = '#E74C3C', linetype = "dashed", linewidth = 0.8) +
  scale_fill_identity() +
  labs(title = 'LTV/CAC Ratio by Acquisition Channel', x = NULL, y = 'LTV/CAC ratio') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

---

## §10 — MRR & MRR Growth Rate

**Formula:**
- MRR = sum of monthly fees for all active subscribers in a month
- MRR growth rate = (MRR_t - MRR_{t-1}) / MRR_{t-1}

In [ ]:
active_subs <- subscriptions |> filter(status == 'active') |> mutate(year_month = floor_date(month, "month"))

mrr <- active_subs |> group_by(year_month) |> summarise(mrr = sum(monthly_fee), .groups = "drop") |>
  arrange(year_month) |> mutate(
    prev_mrr   = lag(mrr),
    mrr_growth = (mrr - prev_mrr) / prev_mrr
  )

print(mrr |> mutate(across(where(is.numeric), ~round(.x, 2))))

# MRR by fee tier
mrr_tier <- active_subs |> group_by(year_month, monthly_fee) |> summarise(subscribers = n_distinct(user_id), .groups = "drop") |>
  mutate(mrr_contribution = monthly_fee * subscribers)
cat("\nMRR by tier:\n")
print(mrr_tier |> group_by(monthly_fee) |> summarise(total_contribution = round(sum(mrr_contribution), 2), .groups = "drop"))

In [ ]:
library(patchwork)

p1 <- ggplot(mrr, aes(x = format(year_month, "%Y-%m"), y = mrr)) +
  geom_col(fill = '#2980B9', color = 'white', width = 0.5) +
  geom_text(aes(label = sprintf("$%s", format(round(mrr), big.mark = ","))), vjust = -0.5, fontface = "bold", size = 3) +
  labs(title = 'Monthly Recurring Revenue (MRR)', x = 'Month', y = 'MRR ($)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

mrr_valid <- mrr |> filter(!is.na(mrr_growth)) |> mutate(bar_color = if_else(mrr_growth >= 0, '#2ECC71', '#E74C3C'))
p2 <- ggplot(mrr_valid, aes(x = format(year_month, "%Y-%m"), y = mrr_growth * 100, fill = bar_color)) +
  geom_col(color = 'white', width = 0.5) +
  geom_hline(yintercept = 0, color = "black", linewidth = 0.4) +
  scale_fill_identity() +
  labs(title = 'MRR Growth Rate (Month-over-Month)', x = 'Month', y = 'Growth rate (%)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

p1 + p2

---

## §11 — CTR (Click-Through Rate)

**Formula:** CTR = clicks / impressions

In [ ]:
# Daily CTR
ctr_daily <- ad_events |> count(event_date, event_type) |>
  pivot_wider(names_from = event_type, values_from = n, values_fill = 0) |>
  mutate(ctr = click / na_if(impression, 0))

cat("Daily CTR (first 5):\n")
print(head(ctr_daily, 5))
cat(sprintf("\nOverall CTR: %.2f%%\n", sum(ctr_daily$click) / sum(ctr_daily$impression) * 100))

# CTR by ad
ctr_ad <- ad_events |> count(ad_id, event_type) |>
  pivot_wider(names_from = event_type, values_from = n, values_fill = 0) |>
  mutate(ctr = click / na_if(impression, 0))
cat("\nCTR by ad:\n")
print(ctr_ad |> arrange(desc(ctr)) |> mutate(ctr = round(ctr, 4)))

In [ ]:
library(slider)
library(patchwork)

ctr_sorted <- ctr_daily |> arrange(event_date) |> mutate(rolling_ctr = slide_dbl(ctr, mean, .before = 6, na.rm = TRUE, .complete = TRUE))

p1 <- ggplot(ctr_sorted, aes(x = event_date, y = rolling_ctr * 100)) +
  geom_ribbon(aes(ymin = 0, ymax = rolling_ctr * 100), alpha = 0.15, fill = '#F39C12', na.rm = TRUE) +
  geom_line(color = '#F39C12', linewidth = 1, na.rm = TRUE) +
  labs(title = 'Daily CTR (7-day rolling avg)', x = 'Date', y = 'CTR (%)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

ctr_ad_sorted <- ctr_ad |> arrange(ctr) |> mutate(ad_id = factor(ad_id, levels = ad_id))
p2 <- ggplot(ctr_ad_sorted, aes(x = ad_id, y = ctr * 100)) +
  geom_col(fill = '#F39C12', color = 'white', alpha = 0.85, width = 0.6) +
  coord_flip() +
  labs(title = 'CTR by Ad', x = NULL, y = 'CTR (%)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

p1 + p2

---

## §12 — GMV (Gross Merchandise Value)

**Formula:** GMV = sum of all completed transaction values in the period

In [ ]:
transactions <- transactions |> mutate(year_month = floor_date(transaction_date, "month"))

gmv_monthly <- transactions |> group_by(year_month) |> summarise(
  total_transactions = n(),
  unique_buyers       = n_distinct(buyer_id),
  gmv                 = sum(transaction_value),
  avg_txn_value       = mean(transaction_value),
  .groups = "drop"
)

print(gmv_monthly |> mutate(across(where(is.numeric), ~round(.x, 2))))
cat(sprintf("\nTotal GMV (Q1 2024): $%s\n", format(round(sum(gmv_monthly$gmv), 2), big.mark = ",")))
cat(sprintf("Avg transaction:     $%s\n", format(round(mean(gmv_monthly$avg_txn_value), 2), big.mark = ",")))

In [ ]:
library(patchwork)

p1 <- ggplot(gmv_monthly, aes(x = format(year_month, "%Y-%m"), y = gmv)) +
  geom_col(fill = '#1ABC9C', color = 'white', width = 0.5) +
  geom_text(aes(label = sprintf("$%s", format(round(gmv), big.mark = ","))), vjust = -0.5, fontface = "bold", size = 3) +
  labs(title = 'Monthly GMV', x = 'Month', y = 'GMV ($)') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

med <- median(transactions$transaction_value)
p2 <- ggplot(transactions, aes(x = transaction_value)) +
  geom_histogram(bins = 50, fill = '#1ABC9C', color = 'white', alpha = 0.85) +
  geom_vline(xintercept = med, color = '#E74C3C', linetype = "dashed", linewidth = 0.8) +
  annotate("text", x = med, y = Inf, label = sprintf("Median = $%.0f", med), vjust = 2, hjust = -0.1, color = '#E74C3C', size = 3) +
  labs(title = 'Transaction Value Distribution', x = 'Transaction value ($)', y = 'Count') +
  theme_minimal() + theme(plot.title = element_text(face = "bold"))

p1 + p2

---

## Quick Reference — tidyverse Cheat Sheet

| Metric | Key tidyverse operation |
|---|---|
| DAU/WAU/MAU | `group_by(period) |> summarise(n_distinct(user_id))` |
| DAU/MAU ratio | join DAU and MAU on period, divide |
| D1/D7/D30 retention | join cohort + activity, `as.numeric(event_date - created_at, units="days")` |
| K-factor | `(sent / inviters) * (accepted / sent)` |
| NPS | `% promoters - % detractors` after `case_when()` binning scores |
| Conversion rate | `n_distinct()` per funnel step, divide |
| LTV | `mean(avg_order_val) * mean(order_count) * mean(lifespan_years)` |
| CAC | `total_spend / new_customers` after joining spend + users |
| LTV/CAC | join LTV and CAC tables, divide |
| MRR | `group_by(month) |> summarise(sum(monthly_fee))` on active subs |
| MRR growth rate | `lag()` and `(mrr - prev) / prev` |
| CTR | `clicks / impressions` after `count() |> pivot_wider()` |
| GMV | `group_by(month) |> summarise(sum(transaction_value))` |